In [ ]:
# M7: PRESCRIPTIVE MAINTENANCE AI
# The orchestrator — gathers all 6 model outputs + ERP data
# and generates an actionable, cost-justified work order

import pandas as pd
import numpy as np
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE FAILURE_GENOME_DB").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
print("Session ready.")

In [ ]:
# Build the CREATE PROCEDURE SQL using string concatenation to avoid $$ parsing issues
proc_body = '''
import json
import numpy as np

def run(session, p_asset_id, p_failure_mode, p_failure_confidence,
        p_rul_hours, p_rul_lower, p_rul_upper, p_degradation_stage,
        p_fatigue_score, p_root_cause, p_risk_level,
        p_sim_intervention_day, p_sim_repair_cost, p_sim_failure_cost):

    asset_id = str(p_asset_id)

    asset_rows = session.sql("SELECT asset_name, asset_type, line_id, manufacturer, model_number FROM FAILURE_GENOME_DB.RAW_OT.ASSET_MASTER WHERE asset_id = '" + asset_id + "'").collect()
    asset = asset_rows[0] if asset_rows else None
    asset_name = str(asset["ASSET_NAME"]) if asset else asset_id
    asset_type = str(asset["ASSET_TYPE"]) if asset else "unknown"
    line_id = str(asset["LINE_ID"]) if asset else "unknown"

    parts_rows = session.sql("SELECT part_id, part_name, quantity_on_hand, lead_time_days, unit_cost FROM FAILURE_GENOME_DB.RAW_IT.PARTS_INVENTORY WHERE ARRAY_CONTAINS('" + asset_id + "'::VARIANT, compatible_assets) ORDER BY unit_cost DESC").collect()

    parts_available = []
    parts_needed = []
    total_parts_cost = 0.0
    parts_in_stock = True

    for p in parts_rows:
        part_info = {"part_id": str(p["PART_ID"]), "part_name": str(p["PART_NAME"]), "quantity_on_hand": int(p["QUANTITY_ON_HAND"]), "lead_time_days": int(p["LEAD_TIME_DAYS"]), "unit_cost": float(p["UNIT_COST"])}
        parts_available.append(part_info)
        fm = str(p_failure_mode).lower()
        pn = str(p["PART_NAME"]).lower()
        needed = False
        if fm == "bearing_wear" and ("bearing" in pn or "seal" in pn):
            needed = True
        elif fm == "misalignment" and ("coupling" in pn or "bearing" in pn):
            needed = True
        elif fm == "imbalance" and ("impeller" in pn or "blade" in pn):
            needed = True
        elif fm == "thermal_degradation" and ("filter" in pn or "separator" in pn or "bearing" in pn):
            needed = True
        if needed:
            parts_needed.append(part_info)
            total_parts_cost += float(p["UNIT_COST"])
            if int(p["QUANTITY_ON_HAND"]) == 0:
                parts_in_stock = False

    shift_rows = session.sql("SELECT shift_id, shift_start, shift_end, operator_name FROM FAILURE_GENOME_DB.RAW_IT.SHIFT_SCHEDULE WHERE line_id = '" + line_id + "' AND shift_start >= CURRENT_TIMESTAMP() AND shift_start <= DATEADD(day, 7, CURRENT_TIMESTAMP()) ORDER BY shift_start LIMIT 5").collect()

    optimal_window = None
    if shift_rows:
        for s in shift_rows:
            start_hour = int(str(s["SHIFT_START"])[11:13]) if len(str(s["SHIFT_START"])) > 13 else 8
            if 6 <= start_hour <= 14:
                optimal_window = {"shift_id": str(s["SHIFT_ID"]), "start": str(s["SHIFT_START"]), "end": str(s["SHIFT_END"]), "operator": str(s["OPERATOR_NAME"])}
                break
        if not optimal_window:
            optimal_window = {"shift_id": str(shift_rows[0]["SHIFT_ID"]), "start": str(shift_rows[0]["SHIFT_START"]), "end": str(shift_rows[0]["SHIFT_END"]), "operator": str(shift_rows[0]["OPERATOR_NAME"])}

    rul = float(p_rul_hours) if p_rul_hours else 999
    stage = int(p_degradation_stage) if p_degradation_stage else 0
    fatigue = float(p_fatigue_score) if p_fatigue_score else 0

    if stage >= 4 or rul < 24:
        urgency = "EMERGENCY"
        wo_type = "emergency"
        action = "Immediate shutdown and emergency repair required"
    elif stage >= 3 or rul < 72 or str(p_risk_level) == "CRITICAL":
        urgency = "URGENT"
        wo_type = "corrective"
        action = "Schedule corrective maintenance within 48 hours"
    elif stage >= 2 or rul < 336 or fatigue > 0.6:
        urgency = "PLANNED"
        wo_type = "corrective"
        action = "Schedule repair during next planned outage window"
    else:
        urgency = "MONITOR"
        wo_type = "preventive"
        action = "Continue monitoring. Add to next PM cycle."

    labor_hours_map = {"bearing_wear": {"compressor": 16, "centrifugal_pump": 12, "electric_motor": 8, "default": 10}, "misalignment": {"centrifugal_pump": 14, "conveyor_drive": 10, "default": 8}, "imbalance": {"centrifugal_pump": 8, "axial_fan": 6, "default": 8}, "thermal_degradation": {"compressor": 6, "electric_motor": 4, "default": 6}}
    fm_map = labor_hours_map.get(str(p_failure_mode), {"default": 8})
    est_labor_hours = fm_map.get(asset_type, fm_map.get("default", 8))
    labor_rate = 100.0
    labor_cost = est_labor_hours * labor_rate

    total_repair_cost = total_parts_cost + labor_cost
    failure_cost = float(p_sim_failure_cost) if p_sim_failure_cost else 50000
    savings = failure_cost - total_repair_cost
    roi = (savings / total_repair_cost * 100) if total_repair_cost > 0 else 0

    work_order = {"wo_type": wo_type, "asset_id": asset_id, "priority": urgency, "status": "pending_approval", "assigned_to": optimal_window["operator"] if optimal_window else "Unassigned", "estimated_labor_hours": est_labor_hours, "estimated_parts_cost": round(total_parts_cost, 2), "estimated_total_cost": round(total_repair_cost, 2), "description": str(p_failure_mode) + " detected on " + asset_name + ". " + str(p_root_cause) + ". RUL: " + str(round(rul)) + "h. Stage: " + str(stage) + "/4."}

    prompt_lines = ["You are a maintenance operations advisor. Generate a concise, actionable maintenance recommendation.", "", "ASSET: " + asset_name + " (" + asset_type + ") on " + line_id, "FAILURE MODE: " + str(p_failure_mode) + " (confidence: " + str(round(float(p_failure_confidence)*100)) + "%)", "ROOT CAUSE: " + str(p_root_cause), "RUL: " + str(round(rul)) + " hours (" + str(round(rul/24, 1)) + " days)", "DEGRADATION STAGE: " + str(stage) + "/4", "RISK LEVEL: " + str(p_risk_level), "URGENCY: " + urgency, "", "PARTS NEEDED: " + (", ".join([p["part_name"] + " ($" + str(p["unit_cost"]) + ", qty: " + str(p["quantity_on_hand"]) + ")" for p in parts_needed]) if parts_needed else "None identified"), "PARTS IN STOCK: " + ("Yes" if parts_in_stock else "No - ORDER IMMEDIATELY"), "REPAIR COST: $" + str(round(total_repair_cost)), "FAILURE COST: $" + str(round(failure_cost)), "SAVINGS: $" + str(round(savings)), "", "Write a 3-4 sentence recommendation for a plant manager. Include action, timeline, cost justification, and risk if delayed. Plain text only."]
    prompt = chr(10).join(prompt_lines)
    safe_prompt = prompt.replace("\'", "\'\'")

    try:
        llm_sql = "SELECT SNOWFLAKE.CORTEX.COMPLETE(\'llama3.3-70b\', \'" + safe_prompt + "\') AS response"
        llm_rows = session.sql(llm_sql).collect()
        recommendation_text = str(llm_rows[0]["RESPONSE"]).strip()
    except Exception as e:
        recommendation_text = action + " " + str(p_root_cause)

    prescription = {"asset_id": asset_id, "asset_name": asset_name, "urgency": urgency, "action": action, "recommendation": recommendation_text, "failure_mode": str(p_failure_mode), "root_cause": str(p_root_cause), "rul_hours": round(rul, 1), "degradation_stage": stage, "parts_needed": parts_needed, "parts_in_stock": parts_in_stock, "optimal_maintenance_window": optimal_window, "cost_analysis": {"repair_cost": round(total_repair_cost, 2), "failure_cost": round(failure_cost, 2), "savings": round(savings, 2), "roi_percent": round(roi, 1)}, "work_order_payload": work_order}

    return prescription
'''

# Use session.sql with $$ delimiters properly
ddl = "CREATE OR REPLACE PROCEDURE FAILURE_GENOME_DB.ML_MODELS.GENERATE_PRESCRIPTION("
ddl += "p_asset_id VARCHAR, p_failure_mode VARCHAR, p_failure_confidence FLOAT, "
ddl += "p_rul_hours FLOAT, p_rul_lower FLOAT, p_rul_upper FLOAT, "
ddl += "p_degradation_stage INTEGER, p_fatigue_score FLOAT, "
ddl += "p_root_cause VARCHAR, p_risk_level VARCHAR, "
ddl += "p_sim_intervention_day INTEGER, p_sim_repair_cost FLOAT, p_sim_failure_cost FLOAT) "
ddl += "RETURNS VARIANT LANGUAGE PYTHON RUNTIME_VERSION = '3.11' "
ddl += "PACKAGES = ('snowflake-snowpark-python', 'numpy') "
ddl += "HANDLER = 'run' EXECUTE AS CALLER AS "
ddl += "'" + proc_body.replace("'", "\\'") + "'"

session.sql(ddl).collect()
print("GENERATE_PRESCRIPTION procedure created.")

In [ ]:
result = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.GENERATE_PRESCRIPTION(
    'ASSET_001',          -- asset_id
    'bearing_wear',       -- failure_mode (from M1)
    0.92,                 -- confidence (from M1)
    72.0,                 -- RUL hours (from M2)
    48.0,                 -- RUL lower CI
    120.0,                -- RUL upper CI
    3,                    -- degradation stage (from M3)
    0.82,                 -- fatigue score (from M4)
    'Progressive outer race spalling due to insufficient lubrication',  -- root cause (from M5)
    'CRITICAL',           -- risk level (from M5)
    3,                    -- intervention day (from M6)
    2500.0,               -- repair cost (from M6)
    578000.0              -- failure cost (from M6)
)
""").collect()

rx = json.loads(result[0][0]) if isinstance(result[0][0], str) else result[0][0]
print("="*70)
print(f"PRESCRIPTION — {rx['asset_name']} [{rx['urgency']}]")
print("="*70)
print(f"\n{rx['recommendation']}")
print(f"\n--- Details ---")
print(f"  Action: {rx['action']}")
print(f"  Failure Mode: {rx['failure_mode']} ({rx['cost_analysis']['roi_percent']:.0f}% ROI)")
print(f"  RUL: {rx['rul_hours']}h | Stage: {rx['degradation_stage']}/4")
print(f"  Parts needed: {', '.join([p['part_name'] for p in rx['parts_needed']])}")
print(f"  Parts in stock: {'Yes' if rx['parts_in_stock'] else 'NO - ORDER NOW'}")
print(f"  Repair cost: ${rx['cost_analysis']['repair_cost']:,.0f}")
print(f"  Failure cost: ${rx['cost_analysis']['failure_cost']:,.0f}")
print(f"  Savings: ${rx['cost_analysis']['savings']:,.0f}")
if rx['optimal_maintenance_window']:
    print(f"  Window: {rx['optimal_maintenance_window']['start']}")
    print(f"  Assigned: {rx['optimal_maintenance_window']['operator']}")

In [ ]:
result2 = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.GENERATE_PRESCRIPTION(
    'ASSET_004',
    'misalignment',
    0.95,
    120.0,
    80.0,
    180.0,
    2,
    0.65,
    'Chronic angular misalignment at pump-motor coupling due to baseplate settlement',
    'HIGH',
    5,
    3200.0,
    156000.0
)
""").collect()

rx2 = json.loads(result2[0][0]) if isinstance(result2[0][0], str) else result2[0][0]
print("="*70)
print(f"PRESCRIPTION — {rx2['asset_name']} [{rx2['urgency']}]")
print("="*70)
print(f"\n{rx2['recommendation']}")
print(f"\n--- Work Order Payload ---")
print(json.dumps(rx2['work_order_payload'], indent=2))

In [ ]:
# This is what the dashboard's "Generate Work Order" button would execute
wo = rx['work_order_payload']
print("="*70)
print("AUTO-GENERATED WORK ORDER SQL")
print("="*70)
print(f"""
INSERT INTO FAILURE_GENOME_DB.RAW_IT.WORK_ORDERS 
(wo_id, asset_id, wo_type, created_date, priority, status, assigned_to, parts_cost, total_cost)
VALUES (
    'WO-AUTO-' || TO_CHAR(CURRENT_TIMESTAMP(), 'YYYYMMDDHH24MISS'),
    '{wo["asset_id"]}',
    '{wo["wo_type"]}',
    CURRENT_TIMESTAMP(),
    '{wo["priority"]}',
    '{wo["status"]}',
    '{wo["assigned_to"]}',
    {wo["estimated_parts_cost"]},
    {wo["estimated_total_cost"]}
);
""")
print("(This SQL would be executed by the dashboard's 'Approve Work Order' button)")

In [ ]:
print("="*70)
print("M7: PRESCRIPTIVE MAINTENANCE AI - COMPLETE")
print("="*70)
print("""
Architecture:
  Inputs from ALL 6 models:
    M1 (failure_mode + confidence) -> What's failing
    M2 (RUL + CI) -> When it will fail
    M3 (stage) -> How far along
    M4 (fatigue_score) -> Hidden degradation level
    M5 (root_cause + risk) -> Why it's failing
    M6 (intervention_day + costs) -> Optimal timing + economics
  
  Plus ERP data:
    - Parts inventory (availability + lead times)
    - Shift schedule (optimal maintenance window)
    - Cost data (labor rates + historical repair costs)

  Output:
    - Natural language recommendation (LLM-generated)
    - Urgency classification (EMERGENCY/URGENT/PLANNED/MONITOR)
    - Parts list with availability check
    - Cost-benefit analysis with ROI
    - Auto-generated work order payload

  Demo flow:
    Judge clicks 'Generate Work Order' on dashboard ->
    Prescriptive AI gathers all model outputs ->
    Returns: 'Replace SKF 6310 bearing on Compressor A1 within 48h.
             Parts in stock. Assign to Day Shift. Cost: $2,500.
             If delayed: $578,000 risk. ROI: 23,020%.'
""")

print("="*70)
print("ALL 7 MODELS COMPLETE")
print("="*70)
print("""
M1: FAILURE_MODE_CLASSIFIER    -> What's wrong
M2: RUL_ESTIMATOR              -> How long until failure
M3: DEGRADATION_STAGER         -> Where on the trajectory
M4: HIDDEN_FATIGUE_DETECTOR    -> Early warning (100 days early)
M5: ROOT_CAUSE_ENGINE          -> Why it's happening (with evidence)
M6: FAILURE_TWIN_SIMULATOR     -> What-if scenarios + cost optimization
M7: PRESCRIPTIVE_AI            -> What to do about it (full work order)

Ready for Phase 4 (Intelligence Layer) and Phase 5 (Cortex Agent).
""")